# Important CNN Designs

Compare a residual block with a depthwise-separable convolution. Check output shapes and parameter counts rather than treating architecture names as guarantees.

In [ ]:
import torch
from torch import nn

torch.manual_seed(35)
torch.set_num_threads(1)
print('CPU | CNN design patterns')

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU(),
                                    nn.Conv2d(channels, channels, 3, padding=1))
    def forward(self, x):
        return torch.relu(x + self.layers(x))

x = torch.randn(2, 16, 12, 12)
block = ResidualBlock(16)
result = block(x)
assert result.shape == x.shape
print('residual input/output:', tuple(x.shape), tuple(result.shape))

In [ ]:
standard = nn.Conv2d(16, 32, 3, padding=1)
depthwise = nn.Conv2d(16, 16, 3, padding=1, groups=16)
pointwise = nn.Conv2d(16, 32, 1)
count = lambda module: sum(parameter.numel() for parameter in module.parameters())
separable_count = count(depthwise) + count(pointwise)
assert count(standard) == 4640
assert separable_count == 704
print('standard 3x3 parameters:', count(standard))
print('depthwise + pointwise parameters:', separable_count)